# Import CSV and text with confidence · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 27. Import CSV and text with confidence

Choose explicit parsing rules, preserve identifiers, and verify imported records.

Before you start: You can inspect a DataFrame, identify missing values, and read a CSV file.

Official reference: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

### M27C01. Inspect a real CSV before analyzing it

The bundled Palmer Penguins CSV is a real public dataset, preserved from its authoritative upstream source. Read the field definitions in data/public/README.md before interpreting its values. The simplified table records 344 observations, eight fields, and source missing values written as NA.

**Read:** What does one row describe, according to the source documentation?

Answer: One observed penguin in this simplified research dataset. The table does not provide an animal ID.

**Predict:** Will the preview contain only complete observations?

Answer: No. The fourth source record lacks its measurements and sex, so missing values are visible in the first five rows.

**Explain:** Why does a successful read_csv call not prove the data are ready for every analysis?

Answer: You still need to check field meaning, units, types, missingness, and which records the question requires.



In [ ]:
# Predict before running: Will the preview contain only complete observations?
path = DATA_DIR / "public" / "penguins.csv"
df = pd.read_csv(path, na_values=["NA"])
result = df.head(5)
print(result)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isna().sum())

**Your turn:** Read only species, island, and body_mass_g using usecols. Store the full three-column table in result and print its shape and the number of known body masses.

For the check, keep these result names: `result`.

**Hint:** Pass the three names as a list to usecols; count() counts known values.

**Watch for this:** These are historical study observations from 2007–2009, not current measurements or a population census.

In [ ]:
# Change this code to complete the task.
# Question: Read only species, island, and body_mass_g using usecols. Store the full three-column table in result and print its shape and the number of known body masses.
path = DATA_DIR / "public" / "penguins.csv"
result = pd.read_csv(path, usecols=["species", "island", "body_mass_g"], na_values=["NA"])
print(result.head())
print("Shape:", result.shape)
print("Known body masses:", result["body_mass_g"].count())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (344, 3)
assert result.columns.tolist() == ["species", "island", "body_mass_g"]
assert result["body_mass_g"].count() == 342
print("Code checks passed. Review the written explanation as well.")

The result has 344 rows and three columns. There are 342 known body masses; two missing measurements remain missing.

**Quick check:** How many body masses support a summary of the available measurements?

1. 344
2. 342
3. 333

Answer: 2. Two of the 344 body-mass entries are missing. The 333 complete-record count answers a different question.

**Explain what you learned:** Inspect the parsed table and its documentation together.

### M27C02. The separator is part of the file contract

A text file does not need a .csv extension to hold a table. Its delimiter determines where fields begin and end. The TSV and TXT files here are documented format conversions of the same public observations, so you can compare the imports directly.

**Read:** Why does the comma-based import have only one column?

Answer: The TSV uses tabs between fields; a comma separator does not split those fields.

**Predict:** What shape should the correctly parsed TSV have?

Answer: 344 rows and eight columns, matching the original CSV.

**Explain:** Does changing a filename extension change its internal delimiter?

Answer: No. Parsing must follow the actual file contents and documented format.



In [ ]:
# Predict before running: What shape should the correctly parsed TSV have?
path = DATA_DIR / "public" / "penguins.tsv"
wrong = pd.read_csv(path, sep=",")
result = pd.read_csv(path, sep="\t", na_values=["NA"])
print("With a comma separator:", wrong.shape)
print("With a tab separator:", result.shape)
print(result.head(3))

**Your turn:** Read penguins.txt into result using its semicolon separator and skip its two comment lines using comment="#". Read penguins.csv into reference and compare the two parsed tables.

For the check, keep these result names: `reference`, `result`.

**Hint:** Use pd.read_csv even though the extension is .txt.

**Watch for this:** Do not use skiprows to hide unexpected malformed records; first identify what those rows mean.

In [ ]:
# Change this code to complete the task.
# Question: Read penguins.txt into result using its semicolon separator and skip its two comment lines using comment="#". Read penguins.csv into reference and compare the two parsed tables.
folder = DATA_DIR / "public"
result = pd.read_csv(folder / "penguins.txt", sep=";", comment="#", encoding="utf-8", na_values=["NA"])
reference = pd.read_csv(folder / "penguins.csv", na_values=["NA"])
pd.testing.assert_frame_equal(result, reference)
print(result.head(3))
print("Same parsed values, columns, and row order as the source CSV.")

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (344, 8)
pd.testing.assert_frame_equal(result, reference)
print("Code checks passed. Review the written explanation as well.")

The comment lines are excluded from the table, and all 344 observations remain. No conversion to an Excel file is necessary to read this text format.

**Quick check:** Which setting separates a TSV into fields?

1. sep=";"
2. sep=","
3. sep="\t"

Answer: 3. TSV means tab-separated values, so the delimiter is a tab.

**Explain what you learned:** Read a table according to its actual delimiter and header rules.

### M27C03. Preserve leading zeros and literal NA codes

This small import fixture is explicitly fictional and separate from the penguin research data. Its IDs contain leading zeros, NA is a literal site code, and MISSING marks one unavailable amount. Encoding, quoting, decimal symbols, and missing-value rules all affect what a reader produces.

**Read:** Why is record_id read as text?

Answer: It is an identifier, not a quantity. Converting 001 to an integer would lose its leading zeros.

**Predict:** Does the NA site code become a missing value in this import?

Answer: No. keep_default_na=False preserves the literal code as text.

**Explain:** Why does the semicolon inside the second note not create an extra column?

Answer: The field is quoted using CSV quoting rules, so the embedded separator belongs to the note.



In [ ]:
# Predict before running: Does the NA site code become a missing value in this import?
path = DATA_DIR / "public" / "import_practice_semicolon.csv"
result = pd.read_csv(path, sep=";", comment="#", encoding="utf-8-sig", dtype="string", keep_default_na=False)
print(result)
print("IDs:", result["record_id"].tolist())
print("Site codes:", result["site_code"].tolist())

**Your turn:** Read the fixture again with record_id and site_code as strings, comma decimal symbols for amount_eur, and only MISSING in amount_eur treated as missing. Keep the NA site code unchanged. Print the known amount count and sum.

For the check, keep these result names: `result`.

**Hint:** Use decimal="," and a column-specific na_values dictionary while keeping keep_default_na=False.

**Watch for this:** The meanings NA=North Area and EU=East Unit apply only to this invented fixture. Do not impose those meanings on other sources.

In [ ]:
# Change this code to complete the task.
# Question: Read the fixture again with record_id and site_code as strings, comma decimal symbols for amount_eur, and only MISSING in amount_eur treated as missing. Keep the NA site code unchanged. Print the known amount count and sum.
path = DATA_DIR / "public" / "import_practice_semicolon.csv"
result = pd.read_csv(path, sep=";", comment="#", encoding="utf-8-sig", dtype={"record_id": "string", "site_code": "string"}, decimal=",", keep_default_na=False, na_values={"amount_eur": ["MISSING"]})
print(result)
print("Known amounts:", result["amount_eur"].count())
print("Known amount sum:", result["amount_eur"].sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["record_id"].tolist() == ["001", "010", "100"]
assert result["site_code"].tolist() == ["NA", "EU", "NA"]
assert result["amount_eur"].count() == 2
assert result["amount_eur"].sum() == 12.5
assert result.loc[2, "amount_eur"] == 0.0
print("Code checks passed. Review the written explanation as well.")

The amounts are 12.5, missing, and 0.0 EUR. Two known amounts sum to 12.5; the missing value was not replaced with zero. Both NA site codes and all leading-zero IDs are preserved.

**Quick check:** Which is a safe way to preserve leading-zero identifiers?

1. Read the identifier column with dtype="string"
2. Convert every column to integers
3. Remove all zeros before importing

Answer: 1. Identifiers need their full text representation even when they contain only digits.

**Explain what you learned:** Specify identifiers, missing tokens, and number formats deliberately.

### M27C04. Verify a CSV round trip with explicit types

A CSV stores text and does not carry a pandas dtype schema. A repeatable import can specify its expected text, integer, and measurement types. Exporting and reading back the table lets you check that values, missingness, and the chosen schema survive.

**Read:** Why use nullable Int64 for body_mass_g?

Answer: The source contains whole-number masses and missing entries; nullable Int64 represents both without inventing values.

**Predict:** How many columns should the exported CSV contain?

Answer: Eight, because index=False does not add the DataFrame index as a ninth field.

**Explain:** Why specify the schema again when reading the exported CSV?

Answer: CSV has no embedded pandas dtype contract, so the reader would otherwise infer types again.



In [ ]:
# Predict before running: How many columns should the exported CSV contain?
schema = {"species": "string", "island": "string", "bill_length_mm": "Float64", "bill_depth_mm": "Float64", "flipper_length_mm": "Int64", "body_mass_g": "Int64", "sex": "string", "year": "Int64"}
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", dtype=schema, na_values=["NA"])
path = OUTPUT_DIR / "penguins_round_trip.csv"
df.to_csv(path, index=False, na_rep="NA")
result = pd.read_csv(path, dtype=schema, na_values=["NA"])
pd.testing.assert_frame_equal(result, df)
print("Rows and columns:", result.shape)
print(result.dtypes)
print("Round-trip values and declared types match.")

**Your turn:** Using the same explicit schema, store only Gentoo observations in gentoo with a reset index. Export gentoo to penguins_gentoo.csv, read the file back into result, and compare result with gentoo.

For the check, keep these result names: `gentoo`, `result`.

**Hint:** Reset the filtered index because the CSV intentionally does not store it.

**Watch for this:** An export should go to OUTPUT_DIR, leaving the downloaded source unchanged.

In [ ]:
# Change this code to complete the task.
# Question: Using the same explicit schema, store only Gentoo observations in gentoo with a reset index. Export gentoo to penguins_gentoo.csv, read the file back into result, and compare result with gentoo.
schema = {"species": "string", "island": "string", "bill_length_mm": "Float64", "bill_depth_mm": "Float64", "flipper_length_mm": "Int64", "body_mass_g": "Int64", "sex": "string", "year": "Int64"}
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", dtype=schema, na_values=["NA"])
gentoo = df.loc[df["species"] == "Gentoo"].reset_index(drop=True)
path = OUTPUT_DIR / "penguins_gentoo.csv"
gentoo.to_csv(path, index=False, na_rep="NA")
result = pd.read_csv(path, dtype=schema, na_values=["NA"])
pd.testing.assert_frame_equal(result, gentoo)
print("Gentoo rows:", len(result))
print("Known body masses:", result["body_mass_g"].count())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (124, 8)
assert result["body_mass_g"].count() == 123
pd.testing.assert_frame_equal(result, gentoo)
print("Code checks passed. Review the written explanation as well.")

The subset contains 124 Gentoo observations, with 123 known body masses. Its row labels are newly created positions, not animal identities.

**Quick check:** Does a CSV automatically preserve a pandas nullable-integer dtype?

1. Yes, every dtype is stored inside CSV
2. No; provide the intended dtype when importing
3. Only if the filename contains Int64

Answer: 2. CSV stores fields as text; the receiving reader applies the type rules.

**Explain what you learned:** Treat an import/export schema as part of reproducibility.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates